# Download do tempo de táxi de saída do (CGNA)

- Painel: <https://portal.cgna.decea.mil.br>
- O relatório é publicado no modo aberto do Power BI pelo próprio CGNA não requerindo chave de aceso.
- Caminho manual para achar o Power BI: Recursos -> Plano de Operações - > Análise Semanal -> Dashboard de Plano de Operações XXX -> Indicadores -> Tempo de Taxi (KPI02 e KPI13) ->  Análise do Tempo de Taxi-out


In [1]:
# pip install pandas
# pip install requests
# pip install pyarrow

In [1]:
import base64
import json

import pandas as pd
import requests

In [2]:
ENDERECO = ("https://app.powerbi.com/view?r=eyJrIjoiY2MwNjMyNjMtMzMzNC00NDlhLTg3"
            "MTctZmE5YjViZmRhOTliIiwidCI6IjI2MjI4ZGNhLTcwZDMtNDkxNy04MjMzLTA4"
            "M2FjMzY1NWE5MSJ9")
CLUSTER = "https://wabi-brazil-south-api.analysis.windows.net"

# O parametro r e um JSON em base64 sem o preenchimento final. A chave "k" e a
# chave publica do relatorio. O modelo muda a cada republicacao do painel, por
# isso e lido do servidor em vez de ficar fixo no codigo.
CHAVE = json.loads(base64.b64decode(ENDERECO.split("r=")[1] + "==="))["k"]
CABECALHO = {"X-PowerBI-ResourceKey": CHAVE}
MODELO = requests.get(f"{CLUSTER}/public/reports/{CHAVE}/modelsAndExploration",
                      headers=CABECALHO, timeout=120).json()["models"][0]["id"]

def download(start, end):
    """Partidas de ICAO entre start e end. Le ICAO e COLUNAS na hora da chamada,
    entao a celula de configuracao precisa ter rodado antes.
    O servidor devolve no maximo 30.000 linhas por consulta."""
    col = lambda p: {"Expression": {"SourceRef": {"Source": "d"}}, "Property": p}
    text = lambda p, v: {"Condition": {"In": {
        "Expressions": [{"Column": col(p)}],
        "Values": [[{"Literal": {"Value": f"'{v}'"}}]]}}}
    date = lambda x: {"Literal": {"Value": f"datetime'{x}T00:00:00'"}}
    # ComparisonKind 2 e ">=" e 3 e "<", com o fim exclusivo para nao repetir o dia 1.
    between = {"Condition": {"And": {
        "Left": {"Comparison": {"ComparisonKind": 2, "Left": {"Column": col("Data")},
                                "Right": date(start)}},
        "Right": {"Comparison": {"ComparisonKind": 3, "Left": {"Column": col("Data")},
                                 "Right": date(end)}}}}}

    pedido = {"modelId": MODELO, "queries": [{"Query": {"Commands": [
        {"SemanticQueryDataShapeCommand": {
            "Query": {"From": [{"Name": "d", "Entity": "dsTaxi"}],
                      "Select": [{"Column": col(c)} for c in COLUNAS],
                      "Where": [text("Aeroporto", ICAO), text("mov", "Dep"), between]},
            # Sem o Groupings a resposta volta 200 porem sem o bloco de dados.
            "Binding": {"Primary": {"Groupings": [
                            {"Projections": list(range(len(COLUNAS)))}]},
                        "DataReduction": {"Primary": {"Window": {"Count": 30000}}}}}}]}}]}

    r = requests.post(f"{CLUSTER}/public/reports/querydata",
                      params={"synchronous": "true"}, headers=CABECALHO,
                      json=pedido, timeout=300)
    r.raise_for_status()
    bloco = r.json()["results"][0]["result"]["data"]["dsr"]["DS"][0]
    dicts = bloco.get("ValueDicts", {})

    # A resposta vem comprimida: "R" marca o bit da coluna que repete a linha
    # anterior, "\u00d8" marca a que e nula, e "C" traz so os valores restantes,
    # em sequencia. Texto repetido vem como indice num dicionario.
    linhas, anterior = [], [None] * len(COLUNAS)
    for pagina in bloco.get("PH", []):
        for item in pagina.get("DM0", []):
            valores, k, linha = item.get("C", []), 0, []
            repete, nulos = item.get("R", 0), item.get("\u00d8", 0)
            for i in range(len(COLUNAS)):
                if nulos >> i & 1:
                    linha.append(None)
                elif repete >> i & 1:
                    linha.append(anterior[i])
                else:
                    v = valores[k] if k < len(valores) else None
                    k += 1
                    d = dicts.get(f"D{i}")
                    if d and isinstance(v, int) and 0 <= v < len(d):
                        v = d[v]
                    linha.append(v)
            anterior = linha
            linhas.append(linha)
    return pd.DataFrame(linhas, columns=COLUNAS)

# 0.2.2 Download

In [3]:
# Variaveis de configuracao do download.
ICAO =      "SBSP"           
INICIO =    pd.Timestamp("2025-01-01 03:00", tz="UTC")
FIM =       pd.Timestamp("2026-01-01 03:00", tz="UTC")

COLUNAS =   ["mov", "indicativo", "pista", "box", "dh_vra", "dh_bimtra",
            "Taxi_Desimp", "Taxi Adicional"]

In [4]:
def window_date(date_start, date_end):
    """Blocos mensais de primeiro ate ultimo, inclusive."""
    while date_start <= date_end:
        next_date = min(date_start + pd.offsets.MonthBegin(1), date_end + pd.Timedelta(days=1))
        yield date_start.strftime("%Y-%m-%d"), next_date.strftime("%Y-%m-%d")
        date_start = next_date

partes = []
for a, b in window_date(INICIO.tz_localize(None).normalize(),
                       FIM.tz_localize(None).normalize()):
    bloco = download(a, b)
    assert len(bloco) < 30000, f"{a} bateu no teto: consulta truncada"
    partes.append(bloco)
    print(f"  {a} -> {b}: {len(bloco):6,}")

df = pd.concat(partes, ignore_index=True)
for c in ["dh_vra", "dh_bimtra"]:
    df[c] = pd.to_datetime(df[c], unit="ms").dt.tz_localize("UTC")
for c in ["Taxi_Desimp", "Taxi Adicional"]:
    df[c] = pd.to_numeric(df[c])

# Mantem apenas registros dentro do intervalo de interesse e ordena por dh_vra
df = df[(df["dh_vra"] > INICIO) & (df["dh_vra"] < FIM)].sort_values("dh_vra").reset_index(drop=True)
# Remove registros duplicados
df = df[df["dh_bimtra"] > df["dh_vra"]].sort_values("dh_vra")
df = df.drop_duplicates(subset=["indicativo", "dh_vra"], keep="first")

  2025-01-01 -> 2025-02-01:  7,954
  2025-02-01 -> 2025-03-01:  7,387
  2025-03-01 -> 2025-04-01:  7,603
  2025-04-01 -> 2025-05-01:  7,481
  2025-05-01 -> 2025-06-01:  7,733
  2025-06-01 -> 2025-07-01:  7,521
  2025-07-01 -> 2025-08-01:  7,867
  2025-08-01 -> 2025-09-01:  7,733
  2025-09-01 -> 2025-10-01:  7,583
  2025-10-01 -> 2025-11-01:  8,062
  2025-11-01 -> 2025-12-01:  7,692
  2025-12-01 -> 2026-01-01:  7,903
  2026-01-01 -> 2026-01-02:    220


# 0.2.3 Gravação 

In [ ]:
df.to_parquet("data/cgh_cgna_2025.parquet", index=False)
df.to_csv("data/cgh_cgna_2025.csv", index=False)

print(f"{len(df):,} partidas")

92,508 partidas
